# DBLP exploration

This tutorial shows how explore DBLP with Gismo.

If you have never used Gismo before, you may want to start with the *Toy example tutorial* or the *ACM* tutorial.

:::note
In previous Gismo version, dblp was handled by Gismo itself. Why we kept the code in the source for legacy purpose, it should be considered as deprecated.

For DBLP access, we now recommmend to use the LDB interface from Gismap.

```python
pip install gismap
```
:::

Recommended requirements to excute this Notebook (after having gismap installed):
- Fast Internet connection (you will need to download a few hundred Mb)
- 4 Gb of free space
- 4 Gb of RAM (8Gb or more recommended)
- Descent CPU (can take more than one hour on slow CPUs)

Here, *documents* are articles in DBLP. The *features* of an article category will vary.

## Initialisation

First, we load the required packages.

In [1]:
import numpy as np
import spacy
from gismo import Corpus, Embedding, CountVectorizer, cosine_similarity, Gismo
from pathlib import Path
from functools import partial

from gismo.post_processing import (
    post_features_cluster_print,
    post_documents_cluster_print,
)

Then, we check that LDB works correctly.

In [2]:
from gismap import LDB

LDB.search_author("Fabien Mathieu")

[LDBAuthor(name='Fabien Mathieu', key='66/2077')]

In [3]:
LDB.author_publications("66/2077")[:4]

[LDBPublication(authors=[LDBAuthor(name='Heger Arfaoui', key='116/4885'), LDBAuthor(name='Pierre Fraigniaud', key='74/3005'), LDBAuthor(name='David Ilcinkas', key='64/5947'), LDBAuthor(name='Fabien Mathieu', key='66/2077')], title='Distributedly Testing Cycle-Freeness.', venue='WG', type='conference', year=2014, key='conf/wg/ArfaouiFIM14'),
 LDBPublication(authors=[LDBAuthor(name='Yacine Boufkhad', key='75/5742'), LDBAuthor(name='Fabien Mathieu', key='66/2077'), LDBAuthor(name='Fabien de Montgolfier', key='57/6313'), LDBAuthor(name='Diego Perino', key='03/3645'), LDBAuthor(name='Laurent Viennot', key='v/LaurentViennot')], title='Fine Tuning of a Distributed VoD System.', venue='ICCCN', type='conference', year=2009, key='conf/icccn/BoufkhadMMPV09'),
 LDBPublication(authors=[LDBAuthor(name='Fabien Mathieu', key='66/2077')], title='Upper Bounds for Stabilization in Acyclic Preference-Based Systems.', venue='SSS', type='conference', year=2007, key='conf/sss/Mathieu07'),
 LDBPublication(aut

By default, this is of the publis are represented internally:

In [4]:
LDB.publis[6666666]

('journals/mcs/ZhangZGF19',
 'A mass-conservative characteristic splitting mixed finite element method for convection-dominated Sobolev equation.',
 'journal',
 [2521603, 3466362, 131057, 2517743],
 'https://doi.org/10.1016/J.MATCOM.2018.12.016',
 ['journals/mcs'],
 '180-191',
 'Math. Comput. Simul.',
 2019)

Each article is a tuple with `key`, `title`, `type`, `authors` (as indices), `url`, `streams`, `pages`, `venue`, `year`. We build a corpus that will tell Gismo that the content of an article is its ``title``.

In [5]:
corpus = Corpus(LDB.publis, to_text=lambda p: p[1])

We build an embedding on top of that corpus.
- We set ``min_df=30`` to exclude rare features;
- We set ``max_df=.02`` to exclude anything present in more than 2% of the corpus;
- We use `spacy` to lemmatize & remove some stopwords; remove `preprocessor=...` from the input if you want to skip this (takes time);
- A few manually selected stopwords to fine-tune things.
- We set ``ngram_range=(1, 2)`` to include bi-grams in the embedding.

This will take a few minutes (without spacy) up to a few hours (with spacy enabled). You can save the embedding for later if you want.

In [6]:
from pathlib import Path

path = Path.home() / "temp"
path.exists()

True

In [7]:
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
keep = {"ADJ", "NOUN", "NUM", "PROPN", "SYM", "VERB"}
vectorizer = CountVectorizer(
    min_df=30,
    max_df=0.02,
    ngram_range=(1, 2),
    dtype=float,
    preprocessor=lambda txt: " ".join(
        [w.lemma_.lower() for w in nlp(txt) if w.pos_ in keep and not w.is_stop]
    ),
    stop_words=[
        "a",
        "about",
        "an",
        "and",
        "for",
        "from",
        "in",
        "of",
        "on",
        "the",
        "with",
    ],
)

try:
    embedding = Embedding.load(filename="dblp_embedding", path=path)
except:
    embedding = Embedding(vectorizer=vectorizer)
    embedding.fit_transform(corpus)
    embedding.dump(filename="dblp_embedding", path=path)

In [8]:
embedding.x

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 86356357 stored elements and shape (8236135, 262705)>

We see from ``embedding.x`` that the embedding links about 7,500,000 documents to 235,000 features. In average, each document is linked to about 10 features.

Now, we initiate the gismo object, and customize post_processers to ease the display.

In [9]:
gismo = Gismo(corpus, embedding)

In [10]:
def post_article(g, i):
    tup = g.corpus[i]
    authors = ", ".join([LDB.author_by_index(k).name for k in tup[3]])
    return f"{tup[1]} By {authors} ({tup[7]}, {tup[8]})"


gismo.post_documents_item = post_article


def post_title(g, i):
    return g.corpus[i][1]
    authors = ", ".join(dic["authors"])
    return f"{tup[1]} By {authors} ({tup[7]}, {tup[8]})"


def post_meta(g, i):
    tup = g.corpus[i]
    authors = ", ".join([LDB.author_by_index(k).name for k in tup[3]])
    return f"{authors} ({tup[7]}, {tup[8]})"


gismo.post_documents_cluster = partial(
    post_documents_cluster_print, post_item=post_title
)
gismo.post_features_cluster = post_features_cluster_print

As the dataset is big, we lower the precision of the computation to speed up things a little bit.

In [11]:
gismo.parameters.n_iter = 2

## Machine Learning (and Covid-19) query

We perform the query *Machine learning*. The returned ``True`` tells that some of the query features were found in the corpus' features.

In [12]:
gismo.rank("Machine Learning")

True

What are the best articles on *Machine Learning*?

In [13]:
gismo.get_documents_by_rank()

['The Changing Landscape of Machine Learning: A Comparative Analysis of Centralized Machine Learning, Distributed Machine Learning and Federated Machine Learning. By Dishita Naik, Nitin Naik (UKCI, 2023)',
 'Resilient Machine Learning for Networked Cyber Physical Systems: A Survey for Machine Learning Security to Securing Machine Learning for CPS. By Felix O. Olowononi, Danda B. Rawat, Chunmei Liu (CoRR, 2021)',
 'Resilient Machine Learning for Networked Cyber Physical Systems: A Survey for Machine Learning Security to Securing Machine Learning for CPS. By Felix O. Olowononi, Danda B. Rawat, Chunmei Liu (IEEE Commun. Surv. Tutorials, 2021)',
 'The Machine Learning Machine: A Tangible User Interface for Teaching Machine Learning. By Magnus Høholt Kaspersen, Karl-Emil Kjær Bilstrup, Marianne Graves Petersen (TEI, 2021)',
 'Exploring fairness and privacy in machine learning. (Exploring fairness and privacy in machine learning). By Carlos Pinzón (None, 2023)',
 'A Machine Learning-oriented

OK, this seems to go everywhere. Maybe we can narrow with a more specific request.

In [14]:
gismo.rank("Machine Learning and covid-19")

True

In [15]:
gismo.get_documents_by_rank()

['Ergonomics of Virtual Learning During COVID-19. By Lu Yuan, Alison Garaudy (AHFE (11), 2021)',
 'University Virtual Learning in Covid Times. By Verónica Marín-Díaz, Eloísa Reche, Javier Martín (Technol. Knowl. Learn., 2022)',
 'Mobile Learning for COVID-19 Prevention. By Zhiyi Wang (EAI Endorsed Trans. e Learn., 2024)',
 'Design Issues in e-Learning during the COVID-19 Pandemic. By Alexandra Hosszu, Cosima Rughinis (CSCS, 2021)',
 'Campus traffic and e-Learning during COVID-19 pandemic. By Thomas Favale, Francesca Soro, Martino Trevisan, Idilio Drago, Marco Mellia (Comput. Networks, 2020)',
 'Campus Traffic and e-Learning during COVID-19 Pandemic. By Thomas Favale, Francesca Soro, Martino Trevisan, Idilio Drago, Marco Mellia (CoRR, 2020)',
 'The Deaf Experience in Remote Learning during COVID-19. By Yosra Bouzid, Mohamed Jemni (ICTA, 2021)',
 'Interpretable Sequence Learning for Covid-19 Forecasting. By Sercan Ö. Arik, Chun-Liang Li, Jinsung Yoon, Rajarishi Sinha, Arkady Epshteyn, Lo

Sounds nice. How are the top-10 articles related? Note: as the graph structure is really sparse on the document side (10 features), it is best to de-activate the query-distortion, which is intended for longer documents.

In [16]:
gismo.parameters.distortion = 0.0
gismo.get_documents_by_cluster(k=10)

 F: 0.46. R: 0.00. S: 0.79.
- F: 0.46. R: 0.00. S: 0.78.
-- F: 0.65. R: 0.00. S: 0.52.
--- Ergonomics of Virtual Learning During COVID-19. (R: 0.00; S: 0.61)
--- University Virtual Learning in Covid Times. (R: 0.00; S: 0.34)
-- Mobile Learning for COVID-19 Prevention. (R: 0.00; S: 0.55)
-- F: 0.72. R: 0.00. S: 0.74.
--- Design Issues in e-Learning during the COVID-19 Pandemic. (R: 0.00; S: 0.66)
--- F: 1.00. R: 0.00. S: 0.71.
---- Campus traffic and e-Learning during COVID-19 pandemic. (R: 0.00; S: 0.71)
---- Campus Traffic and e-Learning during COVID-19 Pandemic. (R: 0.00; S: 0.71)
-- F: 1.00. R: 0.00. S: 0.53.
--- Interpretable Sequence Learning for Covid-19 Forecasting. (R: 0.00; S: 0.53)
--- Interpretable Sequence Learning for COVID-19 Forecasting. (R: 0.00; S: 0.53)
- The Deaf Experience in Remote Learning during COVID-19. (R: 0.00; S: 0.51)
- DeCoP: Deep Learning for COVID-19 Prediction of Survival. (R: 0.00; S: 0.52)


Now, let's look at the main keywords.

In [17]:
gismo.get_features_by_rank(20)

['covid',
 'covid 19',
 '19',
 'learning covid',
 'machine',
 'machine learning',
 'pandemic',
 '19 pandemic',
 'online learning',
 'online',
 '19 detection',
 'chest',
 'student',
 'deep learning',
 'ray',
 'chest ray',
 'prediction',
 'classification',
 'case',
 'ct']

Let's organize them.

In [18]:
# On the feature side, the graph is more dense so we can use query distortion
gismo.get_features_by_cluster(distortion=1)

 F: 0.26. R: 0.06. S: 0.97.
- F: 0.47. R: 0.05. S: 0.97.
-- F: 0.57. R: 0.05. S: 0.97.
--- F: 0.96. R: 0.04. S: 0.97.
---- covid (R: 0.01; S: 1.00)
---- covid 19 (R: 0.01; S: 0.99)
---- 19 (R: 0.01; S: 0.99)
---- learning covid (R: 0.01; S: 0.97)
--- F: 0.98. R: 0.01. S: 0.57.
---- pandemic (R: 0.00; S: 0.58)
---- 19 pandemic (R: 0.00; S: 0.56)
-- F: 0.94. R: 0.00. S: 0.45.
--- online learning (R: 0.00; S: 0.44)
--- online (R: 0.00; S: 0.47)
- F: 1.00. R: 0.01. S: 0.29.
-- machine (R: 0.00; S: 0.29)
-- machine learning (R: 0.00; S: 0.29)
- 19 detection (R: 0.00; S: 0.28)


Rough, very broad analysis:
- One big keyword cluster about Coronavirus / Covid-19, pandemic, online learning;
- Machine Learning as a separate small cluster.

In [19]:
np.dot(gismo.embedding.query_projection("Machine learning")[0], gismo.embedding.y)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 138918 stored elements and shape (1, 8236135)>

139,000 articles with an explicit link to machine learning.

In [20]:
np.dot(gismo.embedding.query_projection("Covid-19")[0], gismo.embedding.y)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 23617 stored elements and shape (1, 8236135)>

23,600 articles with an explicit link to covid-19.

## Authors query

Instead of looking at words, we can explore authors and their collaborations.

 We just have to rewire the corpus to output string of authors.

In [21]:
def to_authors_text(tup):
    return tup[3]


corpus = Corpus(LDB.publis, to_text=to_authors_text)

We can build a new embedding on top of this modified corpus. We tell the vectorizer to be stupid: don't preprocess, words are separated spaces.

This will take a few minutes (you can save the embedding for later if you want).

In [22]:
vectorizer = CountVectorizer(
    dtype=float, preprocessor=lambda x: x, tokenizer=lambda x: x
)
try:
    a_embedding = Embedding.load(filename="dblp_aut_embedding", path=path)
except:
    a_embedding = Embedding(vectorizer=vectorizer)
    a_embedding.fit_transform(corpus)
    a_embedding.dump(filename="dblp_aut_embedding", path=path)

In [23]:
a_embedding.x

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 28298209 stored elements and shape (8236135, 3994027)>

We now have about 4,000,000 authors to explore. Let's reload gismo and try to play.

In [24]:
gismo = Gismo(corpus, a_embedding)
gismo.post_documents_item = post_article
gismo.post_features_item = lambda g, i: LDB.author_by_index(i).name

In [25]:
gismo.post_documents_cluster = partial(
    post_documents_cluster_print, post_item=post_meta
)
gismo.post_features_cluster = post_features_cluster_print

### Laurent Massoulié query

In [26]:
def name_to_index(name):
    return [LDB.keys[LDB.search_author(name)[0].key]]

In [27]:
gismo.rank(name_to_index("Laurent Massoulié"))

True

What are the most central articles of Laurent Massoulié in terms of collaboration?

In [28]:
gismo.get_documents_by_rank(k=10)

['Aligning Embeddings and Geometric Random Graphs: Informational Results and Computational Approaches for the Procrustes-Wasserstein Problem. By Mathieu Even, Luca Ganassali, Jakob Maier, Laurent Massoulié (CoRR, 2024)',
 'Aligning Embeddings and Geometric Random Graphs: Informational Results and Computational Approaches for the Procrustes-Wasserstein Problem. By Mathieu Even, Luca Ganassali, Jakob Maier, Laurent Massoulié (NeurIPS, 2024)',
 'Scalable Local Area Service Discovery. By Richard Black, Heimir Sverrisson, Laurent Massoulié (ICC, 2007)',
 'Asymmetric tree correlation testing for graph alignment. By Jakob Maier, Laurent Massoulié (ITW, 2023)',
 'Asymmetric graph alignment and the phase transition for asymmetric tree correlation testing. By Jakob Maier, Laurent Massoulié (CoRR, 2025)',
 'Decentralized Optimization with Heterogeneous Delays: a Continuous-Time Approach. By Mathieu Even, Hadrien Hendrikx, Laurent Massoulié (CoRR, 2021)',
 'Asynchrony and Acceleration in Gossip Al

We see lots of duplicates. This is not surprising as many articles can published first as a research report, then as a conference paper, last as a journal article. Luckily, Gismo can cover for you.

In [29]:
gismo.get_documents_by_coverage(k=10)

['Aligning Embeddings and Geometric Random Graphs: Informational Results and Computational Approaches for the Procrustes-Wasserstein Problem. By Mathieu Even, Luca Ganassali, Jakob Maier, Laurent Massoulié (CoRR, 2024)',
 'Aligning Embeddings and Geometric Random Graphs: Informational Results and Computational Approaches for the Procrustes-Wasserstein Problem. By Mathieu Even, Luca Ganassali, Jakob Maier, Laurent Massoulié (NeurIPS, 2024)',
 'Scalable Local Area Service Discovery. By Richard Black, Heimir Sverrisson, Laurent Massoulié (ICC, 2007)',
 'Asymmetric tree correlation testing for graph alignment. By Jakob Maier, Laurent Massoulié (ITW, 2023)',
 'Asymmetric graph alignment and the phase transition for asymmetric tree correlation testing. By Jakob Maier, Laurent Massoulié (CoRR, 2025)',
 'Decentralized Optimization with Heterogeneous Delays: a Continuous-Time Approach. By Mathieu Even, Hadrien Hendrikx, Laurent Massoulié (CoRR, 2021)',
 'Asynchrony and Acceleration in Gossip Al

Hum, not working well. The reason here is query distortion. Query distortion is a gismo feature that modulates the clustering with the query. Sadly, when features are authors, the underlying graph has a very specific structure (highly sparse and redundant) that makes query distortion *too* effective. The solution is to desactivate it.

In [30]:
gismo.parameters.distortion = 0
gismo.get_documents_by_coverage(k=10)

['Aligning Embeddings and Geometric Random Graphs: Informational Results and Computational Approaches for the Procrustes-Wasserstein Problem. By Mathieu Even, Luca Ganassali, Jakob Maier, Laurent Massoulié (CoRR, 2024)',
 'Scalable Local Area Service Discovery. By Richard Black, Heimir Sverrisson, Laurent Massoulié (ICC, 2007)',
 'An Impossibility Result for Reconstruction in a Degree-Corrected Planted-Partition Model. By Lennart Gulikers, Marc Lelarge, Laurent Massoulié (CoRR, 2015)',
 'Collective Tree Exploration via Potential Function Method. By Romain Cosson, Laurent Massoulié (ITCS, 2024)',
 'Asymmetric tree correlation testing for graph alignment. By Jakob Maier, Laurent Massoulié (ITW, 2023)',
 'Decentralized Optimization with Heterogeneous Delays: a Continuous-Time Approach. By Mathieu Even, Hadrien Hendrikx, Laurent Massoulié (CoRR, 2021)',
 'Concentration of Non-Isotropic Random Tensors with Applications to Learning and Empirical Risk Minimization. By Mathieu Even, Laurent Ma

Much better. No duplicate and more diversity in the results. Let's observe the communities.

In [31]:
gismo.get_documents_by_cluster(k=20, resolution=0.9)

 F: 0.38. R: 0.06. S: 0.87.
- F: 0.39. R: 0.06. S: 0.86.
-- F: 0.47. R: 0.05. S: 0.81.
--- F: 0.53. R: 0.04. S: 0.79.
---- F: 0.54. R: 0.02. S: 0.70.
----- F: 0.70. R: 0.01. S: 0.60.
------ F: 1.00. R: 0.01. S: 0.49.
------- Mathieu Even, Luca Ganassali, Jakob Maier, Laurent Massoulié (CoRR, 2024) (R: 0.00; S: 0.49)
------- Mathieu Even, Luca Ganassali, Jakob Maier, Laurent Massoulié (NeurIPS, 2024) (R: 0.00; S: 0.49)
------ F: 1.00. R: 0.01. S: 0.60.
------- Jakob Maier, Laurent Massoulié (ITW, 2023) (R: 0.00; S: 0.60)
------- Jakob Maier, Laurent Massoulié (CoRR, 2025) (R: 0.00; S: 0.60)
----- F: 1.00. R: 0.01. S: 0.65.
------ Luca Ganassali, Laurent Massoulié (COLT, 2020) (R: 0.00; S: 0.65)
------ Luca Ganassali, Laurent Massoulié (CoRR, 2020) (R: 0.00; S: 0.65)
---- F: 0.80. R: 0.02. S: 0.69.
----- F: 1.00. R: 0.01. S: 0.60.
------ Mathieu Even, Hadrien Hendrikx, Laurent Massoulié (CoRR, 2021) (R: 0.00; S: 0.60)
------ Mathieu Even, Hadrien Hendrikx, Laurent Massoulié (CoRR, 2020) 

OK! We see that the articles are organized by writing commmunities. Also note how Gismo managed to organize a hierachical grouping of the communities.

Now, let's look in terms of authors. This is actually the interesting part when studying collaborations.

In [32]:
gismo.get_features_by_rank()

['Laurent Massoulié',
 'Marc Lelarge',
 'Mathieu Even',
 'Hadrien Hendrikx',
 'Peter B. Key',
 'Stratis Ioannidis',
 'Nidhi Hegde',
 'Romain Cosson',
 'Francis R. Bach',
 'Luca Ganassali',
 'Anne-Marie Kermarrec',
 'Don Towsley',
 'Ayalvadi Ganesh',
 'Kevin Scaman',
 'Lennart Gulikers',
 'Milan Vojnovic',
 'Dan-Cristian Tomozei',
 'Laurent Viennot',
 'Amin Karbasi']

We see many authors that were not present in the articles listed above. This is an important observation: central articles (with respect to a query) are not necessarily written by central authors!

Let's organize them into communities.

In [33]:
gismo.get_features_by_cluster(resolution=0.6)

 F: 0.01. R: 0.20. S: 0.53.
- F: 0.01. R: 0.20. S: 0.53.
-- F: 0.02. R: 0.20. S: 0.54.
--- F: 0.02. R: 0.19. S: 0.52.
---- F: 0.07. R: 0.15. S: 0.48.
----- F: 0.21. R: 0.12. S: 0.52.
------ F: 0.24. R: 0.11. S: 0.56.
------- Laurent Massoulié (R: 0.10; S: 1.00)
------- Mathieu Even (R: 0.01; S: 0.26)
------ Hadrien Hendrikx (R: 0.01; S: 0.20)
----- F: 0.12. R: 0.02. S: 0.25.
------ Marc Lelarge (R: 0.01; S: 0.14)
------ Luca Ganassali (R: 0.01; S: 0.16)
------ Lennart Gulikers (R: 0.00; S: 0.18)
----- Francis R. Bach (R: 0.01; S: 0.05)
----- Kevin Scaman (R: 0.00; S: 0.10)
----- Milan Vojnovic (R: 0.00; S: 0.05)
---- F: 0.04. R: 0.02. S: 0.16.
----- F: 0.08. R: 0.01. S: 0.14.
------ Peter B. Key (R: 0.01; S: 0.12)
------ Ayalvadi Ganesh (R: 0.01; S: 0.09)
----- Anne-Marie Kermarrec (R: 0.01; S: 0.07)
---- F: 0.03. R: 0.01. S: 0.08.
----- Stratis Ioannidis (R: 0.01; S: 0.09)
----- Amin Karbasi (R: 0.00; S: 0.03)
---- Nidhi Hegde (R: 0.01; S: 0.14)
--- F: 0.06. R: 0.01. S: 0.14.
---- Rom

### Jim Roberts  query

In [34]:
gismo.rank(name_to_index("James W. Roberts"))

True

Let's have a covering set of articles.

In [35]:
gismo.get_documents_by_coverage(k=10)

['Integrated Admission Control for Streaming and Elastic Traffic. By Nabil Benameur, Slim Ben Fredj, Frank Delcoigne, Sara Oueslati, James W. Roberts (QofIS, 2001)',
 'Statistical bandwidth sharing: a study of congestion at flow level. By Slim Ben Fredj, Thomas Bonald, Alexandre Proutière, G. Régnié, James W. Roberts (SIGCOMM, 2001)',
 'An In-Camera Data Stream Processing System for Defect Detection in Web Inspection Tasks. By S. Hossain Hajimowlana, Roberto Muscedere, Graham A. Jullien, James W. Roberts (Real Time Imaging, 1999)',
 "Modifications of Thomae's Function and Differentiability. By Kevin Beanland, James W. Roberts, Craig Stevenson (Am. Math. Mon., 2009)",
 'A Traffic Control Framework for High Speed Data Transmission. By James W. Roberts, Brahim Bensaou, Y. Canetti (Modelling and Evaluation of ATM Networks, 1993)',
 'Broadband Network Teletraffic - Performance Evaluation and Design of Broadband Multiservice Networks: Final Report of Action COST 242 By James W. Roberts, Ugo 

Who are the associated authors?

In [36]:
gismo.get_features_by_rank(k=10)

['James W. Roberts',
 'Thomas Bonald',
 'Sara Oueslati',
 'Maher Hamdi',
 'Jorma T. Virtamo',
 'Ali Ibrahim',
 'Alexandre Proutière',
 'Slim Ben Fredj',
 'Jussi Kangasharju',
 'Keith W. Ross']

Let's organize them.

In [37]:
gismo.get_features_by_cluster(k=10, resolution=0.4)

 F: 0.01. R: 0.24. S: 0.54.
- F: 0.01. R: 0.23. S: 0.54.
-- F: 0.04. R: 0.21. S: 0.53.
--- F: 0.20. R: 0.20. S: 0.53.
---- James W. Roberts (R: 0.14; S: 1.00)
---- Thomas Bonald (R: 0.04; S: 0.22)
---- Sara Oueslati (R: 0.02; S: 0.29)
---- Slim Ben Fredj (R: 0.01; S: 0.29)
--- Alexandre Proutière (R: 0.01; S: 0.03)
-- Maher Hamdi (R: 0.01; S: 0.11)
-- Jorma T. Virtamo (R: 0.01; S: 0.06)
-- Ali Ibrahim (R: 0.01; S: 0.05)
- F: 0.06. R: 0.01. S: 0.03.
-- Jussi Kangasharju (R: 0.00; S: 0.03)
-- Keith W. Ross (R: 0.00; S: 0.02)


### Combined queries

We can input multiple authors.

In [38]:
gismo.rank(name_to_index("Laurent_Massoulié") + name_to_index("James W. Roberts"))

True

Let's have a covering set of articles.

In [39]:
gismo.get_documents_by_coverage(k=10)

['Integrated Admission Control for Streaming and Elastic Traffic. By Nabil Benameur, Slim Ben Fredj, Frank Delcoigne, Sara Oueslati, James W. Roberts (QofIS, 2001)',
 'Statistical bandwidth sharing: a study of congestion at flow level. By Slim Ben Fredj, Thomas Bonald, Alexandre Proutière, G. Régnié, James W. Roberts (SIGCOMM, 2001)',
 'An In-Camera Data Stream Processing System for Defect Detection in Web Inspection Tasks. By S. Hossain Hajimowlana, Roberto Muscedere, Graham A. Jullien, James W. Roberts (Real Time Imaging, 1999)',
 "Modifications of Thomae's Function and Differentiability. By Kevin Beanland, James W. Roberts, Craig Stevenson (Am. Math. Mon., 2009)",
 'A Traffic Control Framework for High Speed Data Transmission. By James W. Roberts, Brahim Bensaou, Y. Canetti (Modelling and Evaluation of ATM Networks, 1993)',
 'Broadband Network Teletraffic - Performance Evaluation and Design of Broadband Multiservice Networks: Final Report of Action COST 242 By James W. Roberts, Ugo 

Note that we get here only articles by Roberts, yet the articles returned have sightly changed.

Now, let's look at the main authors.

In [40]:
gismo.get_features_by_rank()

['James W. Roberts',
 'Laurent Massoulié',
 'Thomas Bonald',
 'Sara Oueslati',
 'Marc Lelarge',
 'Maher Hamdi',
 'Nidhi Hegde',
 'Mathieu Even',
 'Jorma T. Virtamo',
 'Alexandre Proutière',
 'Hadrien Hendrikx',
 'Peter B. Key',
 'Stratis Ioannidis',
 'Ali Ibrahim']

We see a mix of both co-authors. How are they organized?

In [41]:
gismo.get_features_by_cluster(resolution=0.4)

 F: 0.02. R: 0.19. S: 0.57.
- F: 0.03. R: 0.19. S: 0.57.
-- F: 0.03. R: 0.13. S: 0.57.
--- F: 0.20. R: 0.10. S: 0.65.
---- James W. Roberts (R: 0.07; S: 0.93)
---- Thomas Bonald (R: 0.02; S: 0.21)
---- Sara Oueslati (R: 0.01; S: 0.27)
--- Maher Hamdi (R: 0.01; S: 0.10)
--- Nidhi Hegde (R: 0.00; S: 0.07)
--- Jorma T. Virtamo (R: 0.00; S: 0.05)
--- Alexandre Proutière (R: 0.00; S: 0.04)
--- Ali Ibrahim (R: 0.00; S: 0.05)
-- F: 0.17. R: 0.05. S: 0.19.
--- Laurent Massoulié (R: 0.05; S: 0.37)
--- Mathieu Even (R: 0.00; S: 0.10)
--- Hadrien Hendrikx (R: 0.00; S: 0.07)
-- Marc Lelarge (R: 0.01; S: 0.05)
-- Peter B. Key (R: 0.00; S: 0.05)
- Stratis Ioannidis (R: 0.00; S: 0.03)


## Cross-gismo

Gismo can combine two embeddings two create one hybrid gismo. This is called a cross-gismo (XGismo). This features can be used to analyze authors with respect to the words they use (and vice-versa).

In [42]:
from gismo.gismo import XGismo

gismo = XGismo(x_embedding=a_embedding, y_embedding=embedding)
gismo.diteration.n_iter = 2  # to speed up a little bit computation time

Note that XGismo does not use the underlying corpus, so we can now close the source (the source keeps the file ``dblp.data`` open).

In [43]:
gismo.post_documents_item = lambda g, i: LDB.author_by_index(i).name
gismo.post_features_cluster = post_features_cluster_print
gismo.post_documents_cluster = post_documents_cluster_print

Let's try a request.

In [44]:
gismo.rank("self-stabilization")

True

What are the associated keywords?

In [45]:
gismo.get_features_by_rank(k=10)

['stabilization',
 'self',
 'self stabilization',
 'stabilize',
 'self stabilize',
 'distribute',
 'fault',
 'sensor',
 'distributed',
 'nonlinear']

How are keywords structured?

In [46]:
gismo.get_features_by_cluster(k=20, resolution=0.8)

 F: 0.11. R: 0.02. S: 0.80.
- F: 0.81. R: 0.02. S: 0.80.
-- F: 0.94. R: 0.01. S: 0.81.
--- stabilization (R: 0.00; S: 0.80)
--- self (R: 0.00; S: 0.76)
--- self stabilization (R: 0.00; S: 0.82)
--- stabilize (R: 0.00; S: 0.73)
--- self stabilize (R: 0.00; S: 0.70)
--- stabilizing (R: 0.00; S: 0.75)
-- F: 0.89. R: 0.00. S: 0.75.
--- distribute (R: 0.00; S: 0.67)
--- distributed (R: 0.00; S: 0.77)
-- fault (R: 0.00; S: 0.71)
-- F: 0.94. R: 0.00. S: 0.70.
--- sensor (R: 0.00; S: 0.70)
--- wireless (R: 0.00; S: 0.68)
-- F: 0.96. R: 0.00. S: 0.51.
--- byzantine (R: 0.00; S: 0.50)
--- mobile (R: 0.00; S: 0.55)
--- asynchronous (R: 0.00; S: 0.56)
-- optimal (R: 0.00; S: 0.67)
-- robot (R: 0.00; S: 0.46)
- F: 0.58. R: 0.00. S: 0.19.
-- F: 0.60. R: 0.00. S: 0.12.
--- F: 0.73. R: 0.00. S: 0.09.
---- nonlinear (R: 0.00; S: 0.08)
---- delay (R: 0.00; S: 0.09)
--- linear (R: 0.00; S: 0.24)
-- adaptive (R: 0.00; S: 0.48)


Who are the associated researchers?

In [47]:
gismo.get_documents_by_rank(k=10)

['Ted Herman',
 'Shlomi Dolev',
 'Sébastien Tixeuil',
 'Sukumar Ghosh',
 'George Varghese',
 'Shay Kutten',
 'Stéphane Devismes',
 'Toshimitsu Masuzawa',
 'Stefan Schmid',
 'Swan Dubois']

How are they structured?

In [48]:
gismo.get_documents_by_cluster(k=10, resolution=0.9)

 F: 0.72. R: 0.05. S: 0.83.
- F: 0.94. R: 0.05. S: 0.83.
-- F: 0.96. R: 0.01. S: 0.81.
--- Ted Herman (R: 0.01; S: 0.81)
--- George Varghese (R: 0.00; S: 0.80)
-- F: 0.95. R: 0.02. S: 0.77.
--- Shlomi Dolev (R: 0.01; S: 0.78)
--- Stéphane Devismes (R: 0.00; S: 0.77)
--- Toshimitsu Masuzawa (R: 0.00; S: 0.71)
-- F: 0.98. R: 0.01. S: 0.81.
--- Sébastien Tixeuil (R: 0.01; S: 0.82)
--- Sukumar Ghosh (R: 0.00; S: 0.80)
-- Shay Kutten (R: 0.00; S: 0.84)
-- Swan Dubois (R: 0.00; S: 0.82)
- Stefan Schmid (R: 0.00; S: 0.64)


We can also query researchers. Just use underscores in the query and add `y=False` to indicate that the input is *documents*.

In [49]:
gismo.rank(
    name_to_index("Sébastien_Tixeuil") + name_to_index("Fabien_Mathieu"), y=False
)

True

What are the associated keywords?

In [50]:
gismo.get_features_by_rank(k=10)

['p2p',
 'grid',
 'stabilization',
 'byzantine',
 'reloaded',
 'refresh',
 'self',
 'self stabilization',
 'fun',
 'live streaming']

Using covering can yield other keywords of interest.

In [51]:
gismo.get_features_by_coverage(k=10)

['p2p',
 'grid',
 'preference',
 'stabilization',
 'reloaded',
 'refresh',
 'fun',
 'p2p networks',
 'acyclic',
 'streaming']

How are keywords structured?

In [52]:
gismo.get_features_by_cluster(k=20, resolution=0.7)

 F: 0.52. R: 0.19. S: 0.67.
- F: 0.60. R: 0.17. S: 0.67.
-- F: 0.81. R: 0.02. S: 0.40.
--- p2p (R: 0.02; S: 0.38)
--- streaming (R: 0.01; S: 0.37)
-- F: 0.82. R: 0.07. S: 0.58.
--- stabilization (R: 0.01; S: 0.55)
--- byzantine (R: 0.01; S: 0.52)
--- self (R: 0.01; S: 0.59)
--- self stabilization (R: 0.01; S: 0.56)
--- stabilize (R: 0.01; S: 0.58)
--- self stabilize (R: 0.01; S: 0.57)
--- gathering (R: 0.01; S: 0.50)
-- reloaded (R: 0.01; S: 0.39)
-- F: 0.90. R: 0.04. S: 0.35.
--- refresh (R: 0.01; S: 0.33)
--- live streaming (R: 0.01; S: 0.35)
--- pagerank (R: 0.01; S: 0.33)
--- old (R: 0.01; S: 0.34)
--- live (R: 0.01; S: 0.38)
-- fun (R: 0.01; S: 0.50)
-- p2p networks (R: 0.01; S: 0.38)
-- acyclic (R: 0.01; S: 0.38)
- grid (R: 0.01; S: 0.50)
- preference (R: 0.01; S: 0.31)


Who are the associated researchers?

In [53]:
gismo.get_documents_by_rank(k=10)

['Sébastien Tixeuil',
 'Fabien Mathieu',
 'Shlomi Dolev',
 'Michel Raynal',
 'Maria Potop-Butucaru',
 'Athanasios Mazarakis',
 'Toshimitsu Masuzawa',
 'Stéphane Devismes',
 'Fukuhito Ooshita',
 'Elad Michael Schiller']

How are they structured?

In [54]:
gismo.get_documents_by_cluster(k=10, resolution=0.8)

 F: 0.07. R: 0.00. S: 0.62.
- F: 0.57. R: 0.00. S: 0.48.
-- F: 0.86. R: 0.00. S: 0.48.
--- Sébastien Tixeuil (R: 0.00; S: 0.53)
--- Shlomi Dolev (R: 0.00; S: 0.41)
--- Maria Potop-Butucaru (R: 0.00; S: 0.48)
--- Toshimitsu Masuzawa (R: 0.00; S: 0.44)
--- Stéphane Devismes (R: 0.00; S: 0.41)
--- Fukuhito Ooshita (R: 0.00; S: 0.48)
--- Elad Michael Schiller (R: 0.00; S: 0.36)
-- Michel Raynal (R: 0.00; S: 0.36)
- F: 0.30. R: 0.00. S: 0.64.
-- Fabien Mathieu (R: 0.00; S: 0.71)
-- Athanasios Mazarakis (R: 0.00; S: 0.19)
